# Draft Utility Data Audit

Purpose: inspect what the current raw and processed data can support for a FOF8 draft-board utility target. This notebook focuses on availability, leakage boundaries, position coverage, and rating trajectory shape. It should answer: what can we estimate empirically before we decide on a final utility formula?

In [ ]:
import sys
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
while not (ROOT / "fof8-ml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "fof8-core" / "src"))
sys.path.insert(0, str(ROOT / "fof8-ml" / "src"))

from fof8_core.loader import FOF8Loader  # noqa: E402

RAW_ROOT = ROOT / "fof8-gen" / "data" / "raw"
FEATURES_PATH = ROOT / "fof8-ml" / "data" / "processed" / "features.parquet"

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(120)

RAW_ROOT, FEATURES_PATH

## Configuration

Start with a small league/year slice while iterating. Set `LEAGUES = available_leagues` and `MAX_YEARS = None` when you want the full scan.

In [ ]:
available_leagues = sorted(p.name for p in RAW_ROOT.glob("DRAFT*") if p.is_dir())
available_leagues[:10], len(available_leagues)

In [ ]:
LEAGUES = available_leagues[:8]
MAX_YEARS = 20

def years_for(loader: FOF8Loader, max_years: int | None = MAX_YEARS) -> list[int]:
    years = list(range(loader.initial_sim_year, loader.final_sim_year + 1))
    return years if max_years is None else years[:max_years]

LEAGUES

## Processed Feature/Outcome Columns

This checks what the current pipeline already materializes. Utility targets that become stable should eventually be added to this processed frame through `fof8-core` target builders.

In [ ]:
features = pl.read_parquet(FEATURES_PATH)
features.shape

In [ ]:
candidate_outcome_cols = [
    c for c in features.columns
    if any(token in c for token in ["Overall", "Merit", "DPO", "Games", "Hall", "Award", "Success"])
]
features.select(candidate_outcome_cols).describe()

In [ ]:
position_summary = (
    features
    .group_by("Position_Group")
    .agg(
        pl.len().alias("n"),
        pl.col("Top3_Mean_Current_Overall").mean().alias("mean_top3"),
        pl.col("Peak_Overall").mean().alias("mean_peak"),
        (pl.col("Top3_Mean_Current_Overall") > 0).mean().alias("has_top3_rate"),
        pl.col("Positive_Career_Merit_Cap_Share").mean().alias("mean_positive_merit"),
        pl.col("Career_Games_Played").mean().alias("mean_games"),
    )
    .sort("n", descending=True)
)
position_summary

## Talent Target Distribution

Re-check whether `Top3_Mean_Current_Overall` behaves like a zero-inflated target. The current processed frame shows it is mostly continuous with a heavy low-value mass and a very sparse elite tail.


In [ ]:
talent_targets = ["Top3_Mean_Current_Overall", "Peak_Overall"]

for target_col in talent_targets:
    s = features.get_column(target_col).cast(pl.Float64)
    positive = s.filter(s > 0)
    print(f"\n{target_col}")
    print({
        "n": len(s),
        "zero_count": int((s == 0).sum()),
        "zero_rate": float((s == 0).mean()),
        "positive_count": len(positive),
    })
    print("all quantiles", {
        q: float(s.quantile(q))
        for q in [0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1]
    })


In [ ]:
target_col = "Top3_Mean_Current_Overall"
threshold_rows = []
for threshold in [40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90]:
    count = int((features[target_col] >= threshold).sum())
    threshold_rows.append({
        "threshold": threshold,
        "count": count,
        "rate": count / len(features),
    })

pl.DataFrame(threshold_rows)


In [ ]:
bins = [0, 1, 20, 30, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 101]
hist_rows = []
for lo, hi in zip(bins, bins[1:]):
    if lo == 0 and hi == 1:
        label = "0"
        count = features.filter(pl.col(target_col) == 0).height
    else:
        label = f"[{lo},{hi})"
        count = features.filter(
            (pl.col(target_col) >= lo) & (pl.col(target_col) < hi)
        ).height
    hist_rows.append({"bucket": label, "count": count, "rate": count / len(features)})

pl.DataFrame(hist_rows)


In [ ]:
position_target_summary = (
    features
    .group_by("Position_Group")
    .agg(
        pl.len().alias("n"),
        (pl.col(target_col) == 0).mean().alias("zero_rate"),
        pl.col(target_col).mean().alias("mean"),
        pl.col(target_col).median().alias("median"),
        pl.col(target_col).quantile(0.90).alias("p90"),
        pl.col(target_col).quantile(0.95).alias("p95"),
        pl.col(target_col).quantile(0.99).alias("p99"),
        (pl.col(target_col) >= 60).mean().alias("rate_ge_60"),
        (pl.col(target_col) >= 70).mean().alias("rate_ge_70"),
        (pl.col(target_col) >= 80).mean().alias("rate_ge_80"),
    )
    .sort("p95", descending=True)
)
position_target_summary


## Raw Rating Trajectory Availability

`player_ratings_season_*.csv` has exhibition `Current_Overall` snapshots and includes the full draft class in the rookie-year snapshot. That makes it useful for target construction, not features.

In [ ]:
def load_rookie_index(leagues: list[str]) -> pl.DataFrame:
    frames = []
    for league in leagues:
        loader = FOF8Loader(RAW_ROOT, league)
        years = years_for(loader)
        frame = (
            loader.scan_file("rookies.csv")
            .filter(pl.col("Year").is_in(years))
            .select(
                pl.lit(league).alias("Universe"),
                "Year",
                "Player_ID",
                "Position_Group",
                "Grade",
                "Developed",
            )
            .rename({"Year": "Draft_Year"})
            .collect()
        )
        frames.append(frame)
    return pl.concat(frames, how="vertical_relaxed")

def load_exhibition_ratings(leagues: list[str]) -> pl.DataFrame:
    frames = []
    for league in leagues:
        loader = FOF8Loader(RAW_ROOT, league)
        years = years_for(loader)
        frame = (
            loader.scan_file("player_ratings_season_*.csv")
            .filter(pl.col("Year").is_in(years))
            .filter(pl.col("Scouting") == "Exhibition")
            .select(
                pl.lit(league).alias("Universe"),
                "Year",
                "Player_ID",
                "Current_Overall",
            )
            .collect()
        )
        frames.append(frame)
    return pl.concat(frames, how="vertical_relaxed")

rookies = load_rookie_index(LEAGUES)
ratings = load_exhibition_ratings(LEAGUES)
rookies.shape, ratings.shape

In [ ]:
rookie_ratings = (
    rookies.join(ratings, on=["Universe", "Player_ID"], how="left")
    .with_columns((pl.col("Year") - pl.col("Draft_Year") + 1).alias("Career_Year"))
    .filter(pl.col("Career_Year").is_between(1, 8))
)

availability = (
    rookie_ratings
    .group_by("Career_Year")
    .agg(
        pl.len().alias("rating_rows"),
        pl.col("Player_ID").n_unique().alias("players_with_rating"),
        pl.col("Current_Overall").mean().alias("mean_current"),
        pl.col("Current_Overall").quantile(0.9).alias("p90_current"),
    )
    .sort("Career_Year")
)
availability

In [ ]:
rating_by_position_year = (
    rookie_ratings
    .group_by(["Position_Group", "Career_Year"])
    .agg(
        pl.col("Player_ID").n_unique().alias("n"),
        pl.col("Current_Overall").mean().alias("mean_current"),
        pl.col("Current_Overall").quantile(0.75).alias("p75_current"),
        pl.col("Current_Overall").quantile(0.9).alias("p90_current"),
    )
    .filter(pl.col("n") >= 20)
    .sort(["Position_Group", "Career_Year"])
)
rating_by_position_year

## Candidate Empirical Questions

- Replacement level: estimate from active-player or meaningful-starter distributions by position.
- Rating curve: estimate from veteran market cap share by rating and position.
- Rookie-control window: compute value from years 1-4 or 1-5 using rating trajectories.
- Scarcity: compare position-specific rating percentiles and tail frequencies.
- Final hand-set layer: use only for preferences the data cannot know, such as how much you care about years beyond the rookie contract.